# 02 · Stage 1 — benign vs malignant (multimodal)

The most clinically important stage: **sensitivity to malignancy** is the number that matters. Trained over **all 7 diagnoses** mapped to benign/malignant, so it uses every image and stays multimodal.

**Design.** We train **two** models on the *same* EfficientNetB3 backbone so any
performance gap is attributable to the metadata, not a different vision model:

1. **Image-only baseline** — the original project's architecture (the control).
2. **Multimodal late-fusion** — image embedding ⊕ MLP over age/sex/site.

Both are trained on the **full** stage data with **balanced class weights** and
on-the-fly augmentation — no undersampling (the key methodological fix). Models
are saved to `models/` with names the inference `pipeline.py` expects.

---
*Part of: **A Multimodal, Explainable Hierarchical Framework for Skin-Lesion
Classification with Diffusion-Based Augmentation of Rare Malignancies.***
All training/eval logic lives in the tested modules under `src/`; these
notebooks orchestrate them on a Colab T4.

In [ ]:
!pip -q install scikit-learn matplotlib kaggle >/dev/null 2>&1

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

DRIVE_PROJECT = '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024'

LOCAL_PROJECT = '/content/Skin_Cancer_Research_Grant2024'

os.makedirs(LOCAL_PROJECT, exist_ok=True)

folders = ['data']

for folder in folders:
    source = os.path.join(DRIVE_PROJECT, folder)
    destination = os.path.join(LOCAL_PROJECT, folder)

    print(f"Copying {folder}...")

    shutil.copytree(
        source,
        destination,
        dirs_exist_ok=True
    )

    print(f"✓ {folder} copied")

print("\nAll folders copied to Colab!")

In [2]:

import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024'
except Exception:
    PROJECT_DIR = os.path.abspath('..')

SRC       = os.path.join(PROJECT_DIR, 'src')
MODELS_DIR= os.path.join(PROJECT_DIR, 'models')
# DATA_DIR  = '/content/Skin_Cancer_Research_Grant2024/data'
FIG_DIR   = os.path.join(PROJECT_DIR, 'report', 'figures')

SPLIT_CSV = os.path.join(MODELS_DIR, 'ham_split.csv')
PREP_JSON = os.path.join(MODELS_DIR, 'preprocessing.json')
for d in (MODELS_DIR, DATA_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

assert os.path.isdir(SRC), f"src/ not found at {SRC} — set PROJECT_DIR correctly"
sys.path.insert(0, SRC)
print("PROJECT_DIR =", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR = /content/drive/MyDrive/Projects/Skin_Cancer_Research_Grant2024


In [ ]:
import os, json
import pandas as pd
import data as D, metadata as M, model as Model

STAGE = "stage1"
EPOCHS = 30
BATCH = 32

proc = M.MetadataProcessor.load(PREP_JSON)
df = pd.read_csv(SPLIT_CSV)
sdf = D.filter_for_stage(df, STAGE)
tr = sdf[sdf.split == 'train']; va = sdf[sdf.split == 'val']; te = sdf[sdf.split == 'test']
n_classes = len(D.STAGE_CLASSES[STAGE])
class_weight = D.compute_class_weights(tr, STAGE)
print("classes:", D.STAGE_CLASSES[STAGE], "| n_classes:", n_classes)
print("train/val/test:", len(tr), len(va), len(te))
print("class weights:", class_weight)

classes: ['benign', 'malignant'] | n_classes: 2
train/val/test: 7235 1277 1503
class weights: {0: 0.6212433453546282, 1: 2.561968838526912}


In [ ]:
# ---------------- Image-only baseline ----------------
tf_tr = D.make_dataset(tr, STAGE, None, multimodal=False, batch_size=BATCH, training=True, augment=True)
tf_va = D.make_dataset(va, STAGE, None, multimodal=False, batch_size=BATCH, training=False)

baseline = Model.compile_model(Model.build_image_only_baseline(n_classes))
base_path = os.path.join(MODELS_DIR, "stage1_baseline.keras")
hist_b = baseline.fit(tf_tr, validation_data=tf_va, epochs=EPOCHS,
                      class_weight=class_weight,
                      callbacks=Model.make_callbacks(base_path))
baseline.save(base_path)
json.dump(hist_b.history, open(base_path.replace('.keras', '_history.json'), 'w'))
print("saved baseline ->", base_path)

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 450s 1s/step - accuracy: 0.5583 - loss: 0.8039 - val_accuracy: 0.7894 - val_loss: 0.5019 - learning_rate: 1.0000e-04
Epoch 2/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 200s 882ms/step - accuracy: 0.6444 - loss: 0.7418 - val_accuracy: 0.7768 - val_loss: 0.4969 - learning_rate: 1.0000e-04
Epoch 3/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 196s 863ms/step - accuracy: 0.6590 - loss: 0.6950 - val_accuracy: 0.7886 - val_loss: 0.4681 - learning_rate: 1.0000e-04
Epoch 4/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 195s 860ms/step - accuracy: 0.6872 - loss: 0.6622 - val_accuracy: 0.7972 - val_loss: 0.4513 - learning_rate: 1.0000e-04
Epoch 5/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 195s 856ms/step - accuracy: 0.7013 - loss: 0.6551 - val_accuracy: 0.8121 - val_loss: 0.4319 - learning_rate: 1.0000e-04
Epoch 6/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 191s 841ms/step - accuracy: 0.6954 - loss: 0.6447 - val_accuracy: 0.8050 - val_loss: 0.4311 - learning_rate: 5.4

In [ ]:
# ---------------- Multimodal late-fusion ----------------
tf_tr_m = D.make_dataset(tr, STAGE, proc, multimodal=True, batch_size=BATCH, training=True, augment=True)
tf_va_m = D.make_dataset(va, STAGE, proc, multimodal=True, batch_size=BATCH, training=False)

mm = Model.compile_model(Model.build_multimodal_model(n_classes, proc.feature_dim))
mm_path = os.path.join(MODELS_DIR, "stage1_multimodal.keras")
hist_m = mm.fit(tf_tr_m, validation_data=tf_va_m, epochs=EPOCHS,
                class_weight=class_weight,
                callbacks=Model.make_callbacks(mm_path))
mm.save(mm_path)
json.dump(hist_m.history, open(mm_path.replace('.keras', '_history.json'), 'w'))
print("saved multimodal ->", mm_path)

Epoch 1/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 388s 1s/step - accuracy: 0.5718 - loss: 0.7909 - val_accuracy: 0.8121 - val_loss: 0.4970 - learning_rate: 1.0000e-04
Epoch 2/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 182s 801ms/step - accuracy: 0.6401 - loss: 0.7733 - val_accuracy: 0.7980 - val_loss: 0.4598 - learning_rate: 1.0000e-04
Epoch 3/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 192s 841ms/step - accuracy: 0.6676 - loss: 0.7062 - val_accuracy: 0.8089 - val_loss: 0.4506 - learning_rate: 1.0000e-04
Epoch 4/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 189s 832ms/step - accuracy: 0.6836 - loss: 0.6882 - val_accuracy: 0.8113 - val_loss: 0.4338 - learning_rate: 1.0000e-04
Epoch 5/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 187s 825ms/step - accuracy: 0.6796 - loss: 0.6776 - val_accuracy: 0.8128 - val_loss: 0.4248 - learning_rate: 1.0000e-04
Epoch 6/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 186s 817ms/step - accuracy: 0.7121 - loss: 0.6187 - val_accuracy: 0.8066 - val_loss: 0.4270 - learning_rate: 5.4881e-05
Epoch 7/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 181

In [ ]:
# Quick val-accuracy comparison (full metrics come in notebook 06).
print("baseline   best val_acc: %.4f" % max(hist_b.history['val_accuracy']))
print("multimodal best val_acc: %.4f" % max(hist_m.history['val_accuracy']))

baseline   best val_acc: 0.8121
multimodal best val_acc: 0.8128


In [ ]:
# ============================================================
# Print model architectures
# ============================================================

print("\n" + "=" * 100)
print("IMAGE-ONLY BASELINE MODEL")
print("=" * 100)

baseline.summary(
    expand_nested=True,
    show_trainable=True
)


print("\n" + "=" * 100)
print("MULTIMODAL MODEL")
print("=" * 100)

multimodal.summary(
    expand_nested=True,
    show_trainable=True
)